This file allows to analyze results obtained by running NeuralFineGray/experiments_competing_risk.py FRAMINGHAM

In [ ]:
import os 
import pickle
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix

import sys

sys.path.append('../')
sys.path.append('../NeuralFineGray/DeepSurvivalMachines/')
from nfg import datasets

from lifelines.fitters.aalen_johansen_fitter import AalenJohansenFitter as aaj
import matplotlib.ticker as mtick

import matplotlib.pyplot as plt
import seaborn as sns
custom_params = {"axes.spines.right": False, "axes.spines.top": False, "axes.spines.left": False,
                 "axes.spines.bottom": False, "figure.dpi": 300, 'savefig.dpi': 300}
sns.set_theme(style = "whitegrid", rc = custom_params, font_scale = 1.75)

In [ ]:
group_selection = 'sex' # For FRAMINGHAM, a more detailed analysis to split by sex or age

In [ ]:
path = '../Results/' # Path where the data is saved
x, t, e, cNCriates = datasets.load_dataset('FRAMINGHAM', path = '../', competing = True, normalize = False) # Open the data

# Encoding is reversed in DSM so update to have 1 the outcome of interest
e *= 2
e[e == 4] = 1

In [ ]:
times_eval = [365, 5 * 365, 10 * 365] # Time horizons for evaluation

In [ ]:
from metrics import truncated_concordance_td, integrated_brier_score, brier_score as bs

### Utils: The evaluatino metrics used
def evaluate(survival, e = e, t = t, groups = None, times_eval = []):
    folds = survival.iloc[:, -1].values
    survival = survival.iloc[:, :-1]
    survival.columns = pd.MultiIndex.from_frame(pd.DataFrame(index=survival.columns).reset_index().astype(float))
    
    times = survival.columns.get_level_values(1).unique()
    results = {}

    # If multiple risk, compute cause specific metrics
    for r in survival.columns.get_level_values(0).unique():
        for fold in np.arange(5):
            res = {}
            e_train, t_train = e[folds != fold], t[folds != fold]
            e_test,  t_test  = e[folds == fold], t[folds == fold]
            g_train, g_test = (None, None) if groups is None else (groups[folds != fold], groups[folds == fold])            

            survival_fold = survival[folds == fold][r]

            km = (e_train, t_train)
            res['Overall'] = {
                    "BRS": integrated_brier_score(e_test, t_test, 1 - survival_fold.values, times, km = km, competing_risk = int(r))[0],
                }
            
            if len(times_eval) > 0:
                for te in times_eval:
                    ci, km = truncated_concordance_td(e_test, t_test, 1 - survival_fold.values, times, te, km = km, competing_risk = int(r))
                    res[te] = {
                        "CIS": ci,
                        "BRS": bs(e_test, t_test, 1 - survival_fold.values, times, te, km = km, competing_risk = int(r))[0]}

                    for group in groups.unique() if groups is not None else []:
                        km = (e_train[g_train == group], t_train[g_train == group])
                        res["Overall"]["BRS_{}".format(group)], km = integrated_brier_score(e_test[g_test == group], t_test[g_test == group], 1 - survival_fold[g_test == group].values, times, km = km, competing_risk = int(r))

                        res[te]["CIS_{}".format(group)] = truncated_concordance_td(e_test[g_test == group], t_test[g_test == group], 1 - survival_fold[g_test == group].values, times, te, km = km, competing_risk = int(r))[0]
                        res[te]["BRS_{}".format(group)] = bs(e_test[g_test == group], t_test[g_test == group], 1 - survival_fold[g_test == group].values, times, te, km = km, competing_risk = int(r))[0]

                        km = (e_train[g_train != group], t_train[g_train != group])
                        res[te]["Delta_CIS_{}".format(group)] = np.abs(res[te]["CIS_{}".format(group)] - truncated_concordance_td(e_test[g_test != group], t_test[g_test != group], 1 - survival_fold[g_test != group].values, times, te, km = km, competing_risk = int(r))[0])
                        res[te]["Delta_BRS_{}".format(group)] = np.abs(res[te]["BRS_{}".format(group)] - bs(e_test[g_test != group], t_test[g_test != group], 1 - survival_fold[g_test != group].values, times, te, km = km, competing_risk = int(r))[0])

            results[(r, fold)] = pd.DataFrame.from_dict(res)
    results = pd.concat(results)
    results.index.set_names(['Risk', 'Fold', 'Metric'], inplace = True)

    return results

In [ ]:
# To analyze group performance - We did this only for FRAMINGHAM
age = pd.DataFrame(x, columns = cNCriates).AGE
sex = pd.DataFrame(x, columns = cNCriates).SEX.replace({1: 'Male', 2: 'Female'})


if group_selection == 'sex':
    groups = sex
    print(groups.value_counts())
else:
    groups = age
    groups = pd.cut(groups, [0, 50, 100], labels=["50-", "50+"])
    print(groups.value_counts())
for g in groups.unique():
    print("Group {} - Population {} - Outcome {:.2f}% - Competing {:.2f}% - Censoring {:.2f}%".format(g, (groups == g).sum(), 100 * (e[groups == g] == 1).mean(),
                                                                                100 * (e[groups == g] == 2).mean(),100 * (e[groups == g] == 0).mean()))


In [ ]:
for time in times_eval:
    selection = t <= time
    print("Time {} - Outcome {:.2f}% - Competing {:.2f}% - Censoring {:.2f}%".format(time, 100 * (e[selection] == 1).mean(),
                                                                                    100 * (e[selection] == 2).mean(),100 * (e[selection] == 0).mean()))

In [ ]:
# Rename
dict_name = {'nfg': 'NeuralFG', 'nfgnc': 'NeuralFG Non Competing', 
             'finegray': 'Fine-Gray', 'coxcs': 'Cox',
             'dh': 'DeepHit', 'dhnc': 'DeepHit Non Competing',
             'ds': 'DeSurv', 'dsnc': 'DeSurv Non Competing', 
            } 

matched = {
    "NeuralFG": "NeuralFG Non Competing",
    "DeSurv": "DeSurv Non Competing",
    "DeepHit": "DeepHit Non Competing",
    "Fine-Gray": "Cox"
}

In [ ]:
# Open file and compute performance
predictions, results = {}, {}
for file_name in os.listdir(path):
    if 'FRAMINGHAM' in file_name and '.csv' in file_name: 
        model = file_name       
        model = model[model.rindex('_') + 1: model.index('.')]

        if model not in dict_name: continue
        model = dict_name[model]
        
        print("Opening :", file_name, ' - ', model)
        if 'Fine-Gray' in model or 'Cox' in model:
            # Reinitialize index
            predictions[model] = pd.read_csv(path + file_name, header = [0], index_col = 0).T.ffill().T
            index = pd.DataFrame([[i, t] for i in ('1', '2') for t in predictions[model].columns[:100]] + [['Use', '']])
            predictions[model].columns = pd.MultiIndex.from_frame(index)
        else:
            predictions[model] = pd.read_csv(path + file_name, header = [0, 1], index_col = 0)

        results[model] = evaluate(predictions[model], groups = groups, times_eval = times_eval)

results = pd.concat(results).rename(dict_name)
results.index.set_names('Model', level = 0, inplace = True)

In [ ]:
# Compute average performance across fold and models
table = results.groupby(['Model', 'Risk', 'Metric']).apply(lambda x:  pd.Series(["{:.3f} ({:.3f})".format(mean, std) for mean, std in zip(x.mean(), x.std())], index = x.columns))
table = table.unstack(level=-1).stack(level=0).unstack(level=-1).loc[:, ['CIS', 'BRS']]
table = table.reorder_levels(['Risk', 'Model']).sort_index(level = 0, sort_remaining = False)

table

-----

# Split by age

Split overall performance by age or sex (depends on the initial choice)

In [ ]:
method = 'NeuralFG'

In [ ]:
selection = ['Delta_CIS_{}'.format(group) for group in groups.unique()] # Which metric to use

In [ ]:
table = results.groupby(['Model', 'Risk', 'Metric']).apply(lambda x: pd.Series(["{:.3f} ({:.3f})".format(mean, std) for mean, std in zip(x.mean(), x.std())], index = x.columns))
table = table.unstack(level=-1).stack(level=0).loc[[method, matched[method]], selection]
table = table.reorder_levels(['Risk', 'Model', None]).sort_index(level = 0, sort_remaining = False)

difference = (results.loc[method] - results.loc[matched[method]]).groupby(['Risk', 'Metric']).apply(lambda x: pd.Series(["{:.3f} ({:.3f})".format(mean, std) for mean, std in zip(x.mean(), x.std())], index = x.columns))
difference = difference.unstack(level=-1).stack(level=0).loc[:, selection]


In [ ]:
table = table.loc[1].T.stack().reorder_levels([None, 'Metric']).sort_index(level = 0, sort_remaining = False)
table['Difference'] = difference.loc[1].stack()
table

In [ ]:
pd.concat({"Age Group": groups, "Event": pd.Series(e)}, axis = 1).groupby(['Age Group', 'Event']).size().unstack().rename(columns = {0: 'Censoring', 1: 'Death', 2: 'CVD'})

-------

# Risk sets - Guidelines

Explore how this impact the 10-year risk estimation, and associated treatment choice

In [ ]:
labels = ["Untreated", "Treated"]
method = 'NeuralFG'

In [ ]:
# Extract 10 year risk of heart attack
ten_year_survival = {method: 1 - predictions[method]['1'].iloc[:, np.argmin(np.abs(predictions[method]['1'].columns.astype(float) - 3650))], 
                     matched[method]: 1 - predictions[matched[method]]['1'].iloc[:, np.argmin(np.abs(predictions[matched[method]]['1'].columns.astype(float) - 3650))]}

In [ ]:
def estimate_cif(t, e):
    ajf = aaj()
    ajf.fit(t, e, event_of_interest=1)
    return ajf.cumulative_density_.iloc[np.argmin(np.abs(ajf.cumulative_density_.index.astype(float) - 3650))]

In [ ]:
# Selection criterion
older = age >= 40 
women = sex == 'Female'

In [ ]:
events, treated = {}, {}
for group_name, group in zip(['Overall', 'Women', 'Men'], [older, older & women, older & ~women]):
    events[group_name], treated[group_name] = {}, {}
    for model in ten_year_survival:
        risk = ten_year_survival[model][group] # Select in group the number of events

        # Extract the estimated proportion of event by 10 years
        cvd = estimate_cif(t[risk.index], e[risk.index]).values[0] 
        events[group_name][model] = {
            'CVD': cvd * group.sum(),
            'No CVD': (1 - cvd) * group.sum()
        }

        # Extract the estimated propotion of event for patient treated
        treat = estimate_cif(t[risk[risk >= 0.1].index], e[risk[risk >= 0.1].index]).values[0]
        treated[group_name][model] = {
            'CVD': treat * (risk >= 0.1).sum(),
            'No CVD': (1 - treat) * (risk >= 0.1).sum()
        }


In [ ]:
fig, axes = plt.subplots(1, 3, sharey = True, sharex = True, figsize = (14, 4))

for group, ax in zip(events, axes):
    pd.DataFrame(events[group])[[method, matched[method]]].plot.barh(ax = ax, legend = False, color = ['tab:blue', 'tab:red'])
    pd.DataFrame(treated[group])[[method, matched[method]]].plot.barh(ax = ax, alpha = 0.5, color = 'k', legend = False, hatch = '//')
    for cont in ax.containers[:3]:
        for rect, mode in zip(cont.get_children(), events[group][cont.get_label()]):
            x = rect.get_x() + rect.get_width()
            y = rect.get_y() + rect.get_height() / 2
            percentage_text = '{:.1%}'.format(treated[group][cont.get_label()][mode] / older.sum()) # Patient in that group divided by total number
            
            ax.annotate(percentage_text,
                        xy=(x + 3, y),  # 1 unit to the right to avoid overlap
                        ha='left', va='center',
                        fontsize='small', color='black')
    ax.set_title(group)

h, l = ax.get_legend_handles_labels()
l = l[:3] + ['Treated']
axes[-1].legend(h[:-1], l, bbox_to_anchor=(1, 1), loc='upper left', frameon=False)
axes[1].set_xlabel('Number Patients')